In [0]:
import random
from datetime import date
from pyspark.sql import Row
from pyspark.sql import functions as F

catalog, schema = "workspace", "patient_notes"
base_path      = f"/Volumes/{catalog}/{schema}/raw_notes"
patients_table = f"{catalog}.{schema}.patients"

condition_forms = {
    "pneumonia":               ["pneumonia", "Pneumonia", "PNA"],
    "type 2 diabetes":         ["type 2 diabetes", "T2DM", "DM2", "diabetes mellitus type 2"],
    "hypertension":            ["hypertension", "HTN", "high blood pressure"],
    "asthma":                  ["asthma", "Asthma"],
    "lung tumor":              ["lung tumor", "lung mass", "pulmonary neoplasm"],
    "chronic kidney disease":  ["chronic kidney disease", "CKD"],
    "malaria":                 ["malaria", "Malaria"],
    "urinary tract infection": ["urinary tract infection", "UTI"],
    "migraine":                ["migraine", "Migraine"],
    "copd":                    ["COPD", "copd", "chronic obstructive pulmonary disease"],
    "fatty liver":             ["fatty liver", "hepatic steatosis", "NAFLD"],
    "appendicitis":            ["appendicitis", "acute appendicitis"],
    "gallstones":              ["gallstones", "cholelithiasis"],
}
conditions = list(condition_forms)
surgical   = {"lung tumor", "appendicitis", "gallstones"}      # only these can be 'operated'

phrases = {
    "current":      ["presents with {c}", "diagnosed with {c}", "currently being treated for {c}"],
    "pre_existing": ["known case of {c}"],
    "history":      ["history of {c}", "past medical history significant for {c}"],
    "operated":     ["operated for {c}", "s/p surgery for {c}", "status post resection of {c}"],
    "ruled_out":    ["{c} ruled out", "negative for {c}", "no evidence of {c}"],
}

# conditions more likely at certain ages (unlisted conditions get weight 1)
age_weights = {
    "young":  {"asthma": 3, "migraine": 3, "appendicitis": 3, "urinary tract infection": 2, "pneumonia": 2},
    "middle": {"hypertension": 3, "type 2 diabetes": 3, "fatty liver": 3, "gallstones": 3, "migraine": 2},
    "older":  {"hypertension": 3, "type 2 diabetes": 3, "chronic kidney disease": 3, "copd": 3,
               "lung tumor": 2, "pneumonia": 2},
}

complaints = ["fever and cough", "shortness of breath", "headache", "fatigue", "chest pain",
              "abdominal pain", "digestive issues"]

filler = ["Patient was seen and examined at the bedside.",
          "Vital signs remained stable throughout the stay.",
          "No acute events were reported overnight.",
          "Patient tolerated a regular diet.",
          "Care plan was discussed with the patient and family.",
          "Patient is ambulating independently.",
          "Pain is well controlled on the current regimen.",
          "Patient was counseled on warning signs and when to return."]

In [0]:
random.seed(7)
patients = []
for i in range(1, 31):
    age   = random.randint(18, 88)
    group = "young" if age < 35 else ("middle" if age < 60 else "older")
    patients.append(Row(patient_id=f"P{i:04d}", age=age, sex=random.choice(["F", "M"]), age_group=group))

spark.createDataFrame(patients).write.mode("overwrite").saveAsTable(patients_table)
patient_lookup = {p.patient_id: p for p in patients}
display(spark.table(patients_table).groupBy("age_group").count())

In [0]:
def pick_conditions(group, k):
    weights = [age_weights[group].get(c, 1) for c in conditions]
    chosen = []
    while len(chosen) < k:
        c = random.choices(conditions, weights=weights)[0]
        if c not in chosen:
            chosen.append(c)
    return chosen

def pick_status(c):
    return random.choice([s for s in phrases if s != "operated" or c in surgical])   # surgical-only rule

def sentence(c, status):
    s = random.choice(phrases[status]).format(c=random.choice(condition_forms[c]))
    return s[0].upper() + s[1:] + "."

def make_note(note_id, patient, note_date, long_note):
    chosen = pick_conditions(patient.age_group, random.randint(2, 4) if long_note else random.randint(1, 3))
    labels, sentences = [], []
    for c in chosen:
        status = pick_status(c)
        sentences.append(sentence(c, status))
        labels.append(Row(condition=c, status=status))
    weeks = random.randint(1, 4)
    plan = f"Plan: follow up in {weeks} week{'s' if weeks > 1 else ''}."
    if long_note:
        sex = "female" if patient.sex == "F" else "male"
        course = " ".join(f"Day {d}: " + " ".join(random.sample(filler, 3)) for d in range(1, random.randint(2, 5) + 1))
        text = (f"Chief complaint: {random.choice(complaints)}.\n"
                f"History of present illness: {patient.age}-year-old {sex} admitted for evaluation. "
                + " ".join(random.sample(filler, 4)) + "\n"
                f"Hospital course: {course}\n"
                f"Assessment: {' '.join(sentences)}\n"
                f"{plan}")
        note_type = "discharge summary"
    else:
        text = (f"Chief complaint: {random.choice(complaints)}.\n"
                f"Assessment: {' '.join(sentences)}\n"
                f"{plan}")
        note_type = random.choice(["progress note", "consult note"])
    note = Row(note_id=note_id, patient_id=patient.patient_id, note_date=str(note_date),
               note_type=note_type, note_text=text)
    return note, labels

In [0]:
def generate_day(day, n_notes=60, long_share=0.15):
    random.seed(day.toordinal())
    notes, truth = [], []
    for n in range(n_notes):
        note_id = f"N{day:%Y%m%d}-{n:03d}"
        patient = patient_lookup[random.choice(list(patient_lookup))]
        note, labels = make_note(note_id, patient, day, long_note=random.random() < long_share)
        notes.append(note)
        truth.append(Row(note_id=note_id, labels=labels))
    spark.createDataFrame(notes).write.mode("overwrite").json(f"{base_path}/incoming/{day}")
    spark.createDataFrame(truth).write.mode("overwrite").json(f"{base_path}/ground_truth/{day}")
    print(f"{day}: wrote {n_notes} notes")

generate_day(date(2026, 11, 1))
generate_day(date(2026, 11, 2))

In [0]:
reader = spark.read.option("recursiveFileLookup", "true").option("pathGlobFilter", "*.json")
notes = reader.json(f"{base_path}/incoming")
truth = reader.json(f"{base_path}/ground_truth")

display(notes.groupBy("note_type").count())                                     # mostly short, some long
display(truth.select(F.explode("labels").alias("l")).groupBy("l.status").count())  # all 5 statuses
display(truth.select(F.explode("labels").alias("l"))
             .filter("l.status = 'operated'").groupBy("l.condition").count())     # only surgical conditions
display(notes.filter("note_type = 'discharge summary'").select("note_id", "note_text").limit(2))